In [4]:
import importlib, env
importlib.reload(env)
from env import CloudClusterEnv, STEPS_PER_WEEK, MIN_PODS, MAX_PODS

import json
import numpy as np
import torch
import torch.nn as nn
from torch.distributions import Normal

stats = json.load(open('trace_params.json'))['stats']
print("Setup ready.")

Setup complete. Steps per week: 672
CloudClusterEnv defined.
Setup ready.


In [5]:
class ActorCritic(nn.Module):
    def __init__(self, state_dim=32, action_dim=1):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(state_dim, 256), nn.Tanh(),
            nn.Linear(256, 256),       nn.Tanh(),
        )
        self.actor_mean = nn.Linear(256, action_dim)
        self.log_std = nn.Parameter(torch.zeros(action_dim))
        self.critic = nn.Linear(256, 1)

    def forward(self, state):
        x = self.shared(state)
        return self.actor_mean(x), self.critic(x)

    def get_action(self, state):
        mean, value = self.forward(state)
        std = torch.exp(self.log_std)
        dist = Normal(mean, std)
        action = dist.sample()
        log_prob = dist.log_prob(action).sum(-1)
        return action, log_prob, value.squeeze(-1)

    def evaluate_actions(self, state, action):
        mean, value = self.forward(state)
        std = torch.exp(self.log_std)
        dist = Normal(mean, std)
        log_prob = dist.log_prob(action).sum(-1)
        entropy = dist.entropy().sum(-1)
        return log_prob, value.squeeze(-1), entropy

print("ActorCritic defined.")

ActorCritic defined.


In [6]:
def compute_gae(rewards, values, dones, next_value, gamma=0.99, lam=0.95):
    advantages = []
    gae = 0.0
    values = values + [next_value]
    for t in reversed(range(len(rewards))):
        delta = rewards[t] + gamma * values[t+1] * (1 - dones[t]) - values[t]
        gae = delta + gamma * lam * (1 - dones[t]) * gae
        advantages.insert(0, gae)
    returns = [a + v for a, v in zip(advantages, values[:-1])]
    return advantages, returns

def ppo_update(net, optimizer, states, actions, old_log_probs, advantages, returns,
               clip_eps=0.2, epochs=4, value_coef=0.5, entropy_coef=0.01):
    advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
    for _ in range(epochs):
        new_log_probs, values, entropy = net.evaluate_actions(states, actions)
        ratio = torch.exp(new_log_probs - old_log_probs)
        surr1 = ratio * advantages
        surr2 = torch.clamp(ratio, 1-clip_eps, 1+clip_eps) * advantages
        actor_loss = -torch.min(surr1, surr2).mean()
        critic_loss = ((values - returns)**2).mean()
        loss = actor_loss + value_coef*critic_loss - entropy_coef*entropy.mean()
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(net.parameters(), 0.5)
        optimizer.step()
    return actor_loss.item(), critic_loss.item(), entropy.mean().item()

def train(env_instance, net, optimizer, total_steps=250_000, rollout_len=2048):
    state, _ = env_instance.reset()
    state = torch.tensor(state, dtype=torch.float32)
    steps_done = 0
    episode_reward = 0
    episode_rewards = []
    while steps_done < total_steps:
        states, actions, log_probs = [], [], []
        rewards, values, dones = [], [], []
        for _ in range(rollout_len):
            with torch.no_grad():
                action, log_prob, value = net.get_action(state.unsqueeze(0))
            a = action.squeeze(0).numpy()
            next_state, reward, done, tr, info = env_instance.step(a)
            states.append(state); actions.append(action.squeeze(0))
            log_probs.append(log_prob.squeeze(0)); rewards.append(float(reward))
            values.append(value.item()); dones.append(1.0 if done else 0.0)
            episode_reward += reward
            state = torch.tensor(next_state, dtype=torch.float32)
            steps_done += 1
            if done:
                episode_rewards.append(episode_reward); episode_reward = 0
                state, _ = env_instance.reset()
                state = torch.tensor(state, dtype=torch.float32)
        with torch.no_grad():
            _, _, next_value = net.get_action(state.unsqueeze(0))
            next_value = next_value.item()
        advantages, returns = compute_gae(rewards, values, dones, next_value)
        b_states = torch.stack(states); b_actions = torch.stack(actions)
        b_old_lp = torch.stack(log_probs)
        b_adv = torch.tensor(advantages, dtype=torch.float32)
        b_ret = torch.tensor(returns, dtype=torch.float32)
        ppo_update(net, optimizer, b_states, b_actions, b_old_lp, b_adv, b_ret)
        recent = np.mean(episode_rewards[-5:]) if episode_rewards else float('nan')
        print(f"steps {steps_done:6d} | recent ep reward {recent:8.1f}")
    return episode_rewards

print("Training functions defined.")

Training functions defined.


In [7]:
class ConstrainedCloudEnv(CloudClusterEnv):
    """±1 VM/step variant. env.py NOT modified — fully isolated."""
    def step(self, action):
        self._update_hints()
        delta = int(round(float(action[0]) * 1))   # ±1 VM/step (was ±5)
        self.active_vms = int(np.clip(self.active_vms + delta, MIN_PODS, MAX_PODS))
        self.queue.extend(self._generate_jobs())
        jobs_processed, avg_cpu, max_cpu, avg_mem, max_mem = self._assign_jobs()
        breaches = self._check_deadlines()
        self.total_breaches += breaches
        cost = self.active_vms / MAX_PODS
        self.cost_total += cost
        utilisation = max(avg_cpu, avg_mem)
        jobs_due = jobs_processed + breaches
        breach_rate = breaches / jobs_due if jobs_due > 0 else 0.0
        reward = (- self.lambda_cost * cost - self.lambda_sla * breach_rate
                  + self.lambda_util * utilisation)
        self.history.append([avg_cpu, max_cpu, avg_mem, max_mem])
        self.history.pop(0)
        self.prev_queue_len = len(self.queue)
        self.step_count += 1
        obs = self._build_state()
        done = self.step_count >= STEPS_PER_WEEK
        info = {'cost': cost, 'breaches': breaches, 'utilisation': utilisation,
                'active_vms': self.active_vms, 'queue': len(self.queue),
                'avg_cpu': avg_cpu, 'avg_mem': avg_mem, 'hint_active': self.hint_active}
        return obs, reward, done, False, info

print("ConstrainedCloudEnv defined — isolated.")

ConstrainedCloudEnv defined — isolated.


In [8]:
env_c = ConstrainedCloudEnv(stats, enable_surges=True, enable_hints=True,
                            min_surges=8, max_surges=12, seed=None)
net_c = ActorCritic()
optimizer = torch.optim.Adam(net_c.parameters(), lr=3e-4)
print("Training constrained hint-aware agent (250k steps)...\n")
episode_rewards = train(env_c, net_c, optimizer, total_steps=250_000)
torch.save(net_c.state_dict(), 'ppo_hint_constrained.pth')
print(f"\nSaved. First: {episode_rewards[0]:.1f}  Last: {episode_rewards[-1]:.1f}")

Training constrained hint-aware agent (250k steps)...

steps   2048 | recent ep reward   -197.8
steps   4096 | recent ep reward   -205.7
steps   6144 | recent ep reward   -193.8
steps   8192 | recent ep reward   -189.6
steps  10240 | recent ep reward   -204.9
steps  12288 | recent ep reward   -210.8
steps  14336 | recent ep reward   -174.1
steps  16384 | recent ep reward   -175.8
steps  18432 | recent ep reward   -194.9
steps  20480 | recent ep reward   -196.5
steps  22528 | recent ep reward   -197.1
steps  24576 | recent ep reward   -189.8
steps  26624 | recent ep reward   -181.7
steps  28672 | recent ep reward   -204.3
steps  30720 | recent ep reward   -217.3
steps  32768 | recent ep reward   -204.7
steps  34816 | recent ep reward   -215.1
steps  36864 | recent ep reward   -197.1
steps  38912 | recent ep reward   -181.3
steps  40960 | recent ep reward   -204.3
steps  43008 | recent ep reward   -235.8
steps  45056 | recent ep reward   -224.7
steps  47104 | recent ep reward   -226.6
st

In [9]:
# load the constrained hint-aware agent
net_c = ActorCritic()
net_c.load_state_dict(torch.load('ppo_hint_constrained.pth'))
net_c.eval()

def eval_constrained(net, n_episodes, force_zero_hints, seed_base=5000):
    breaches_list = []
    for i in range(n_episodes):
        e = ConstrainedCloudEnv(stats, enable_surges=True, enable_hints=True,
                                min_surges=8, max_surges=12, seed=seed_base+i)
        obs, _ = e.reset()
        obs = torch.tensor(obs, dtype=torch.float32)
        ep = 0
        for t in range(STEPS_PER_WEEK):
            if force_zero_hints:
                obs[-3:] = 0.0
            with torch.no_grad():
                mean, _ = net.forward(obs.unsqueeze(0))
            obs, r, done, tr, info = e.step(mean.squeeze(0).numpy())
            obs = torch.tensor(obs, dtype=torch.float32)
            ep += info['breaches']
            if done: break
        breaches_list.append(ep)
    return np.mean(breaches_list), np.std(breaches_list)

N = 30
with_h, wstd    = eval_constrained(net_c, N, force_zero_hints=False)
without_h, ostd = eval_constrained(net_c, N, force_zero_hints=True)

print("="*54)
print("CONSTRAINED (±1 VM/step) HINT EXPERIMENT — 30 weeks, same surges")
print("="*54)
print(f"  WITH hints active:       {with_h:>8.0f} breaches/week")
print(f"  WITHOUT hints (blanked): {without_h:>8.0f} breaches/week")
print("="*54)
if without_h > 0:
    print(f"\n  Hints reduce breaches by {(without_h-with_h)/without_h*100:.1f}%")

CONSTRAINED (±1 VM/step) HINT EXPERIMENT — 30 weeks, same surges
  WITH hints active:          77971 breaches/week
  WITHOUT hints (blanked):    78010 breaches/week

  Hints reduce breaches by 0.1%
